In [ ]:
# Source Code 9 
# Script to plot GeoTIFF images that belong to the same group on a common canvas.

!pip install rasterio          # Installs the 'rasterio' package for working with raster data

import numpy as np             # Imports NumPy for numerical computations
import os                      # Imports os for interacting with the operating system
import pandas as pd            # Imports pandas for data manipulation and analysis
import rasterio                # Imports rasterio for reading and writing raster datasets
import time                    # Imports time for tracking execution time or delays
from google.colab import drive # Imports Colab's drive module to access Google Drive files

# Mount Google Drive
drive.mount('/content/drive')

# File path
file_path = r'/content/drive/MyDrive/ne/mhp-inspected-duplicate-cleaned-grouped.csv'

# Columns to preserve
columns_to_preserve = ['north_w', 'south_w', 'east_w', 'west_w', 'group_id']

# Read the CSV file and select the desired columns
df = pd.read_csv(file_path, usecols=columns_to_preserve)

# Group by 'group_id' and summarize the variables
summary_data = df.groupby('group_id').agg(
    max_north_w=pd.NamedAgg(column='north_w', aggfunc='max'),
    min_south_w=pd.NamedAgg(column='south_w', aggfunc='min'),
    max_east_w=pd.NamedAgg(column='east_w', aggfunc='max'),
    min_west_w=pd.NamedAgg(column='west_w', aggfunc='min')
).reset_index()

print(summary_data)

import re

def extract_group_id(file_name):
    match = re.search(r'group(\d+)', file_name)
    if match:
        group_id = int(match.group(1))
        return group_id
    else:
        return None

# Directory containing images
image_directory = '/content/drive/MyDrive/ne/mhp-inspected-duplicate-cleaned-grouped-overlayed-thresholded/'

# Get a list of all image files in the directory
image_files = [f for f in os.listdir(image_directory) if f.endswith('.png')]

# Iterate over each image file
for image_file in image_files:
    # Image file path
    image_path = os.path.join(image_directory, image_file)

    # Extract group_id from the image file name
    file_name = image_file.split('/')[-1]
    group_id = int(float(extract_group_id(file_name)))

    # Create a new GeoTIFF file
    output_directory = image_directory + '-geotiff'
    output_path = os.path.join(output_directory, f'group{group_id}.tif')

    # Find lon_max, lon_min, lat_max, and lat_min using the group_id
    lon_max = summary_data[summary_data['group_id'] == group_id]['max_east_w'].values[0]
    lon_min = summary_data[summary_data['group_id'] == group_id]['min_west_w'].values[0]
    lat_max = summary_data[summary_data['group_id'] == group_id]['max_north_w'].values[0]
    lat_min = summary_data[summary_data['group_id'] == group_id]['min_south_w'].values[0]

    # Read the image
    image = rasterio.open(image_path)

    # Get image width and height in pixels
    image_width = image.width
    image_height = image.height

    # Calculate the spatial resolution
    resolution_x = (lon_max - lon_min) / image_width
    resolution_y = (lat_max - lat_min) / image_height

    # Calculate the affine transformation matrix
    transform = rasterio.Affine(resolution_x, 0, lon_min,
                               0, -resolution_y, lat_max)

    # Read the RGB image data
    image_data = image.read()  # Read all bands

    # Create an empty array for the new GeoTIFF
    new_image = np.zeros((image_height, image_width), dtype=image_data.dtype)

    # Assign the image data to the new GeoTIFF array
    new_image[:, :] = image_data[:, :]

    # ... (previous code)
    # Get CRS in rasterio format
    crs = rasterio.crs.CRS.from_epsg(4326)

    # Write the new GeoTIFF file from RGB input
    with rasterio.open(output_path, 'w', driver='GTiff', width=image_width,
                   height=image_height, count=image.count, dtype=image_data.dtype,
                   crs=crs, transform=transform) as dst:
        for band in range(image.count):
            dst.write(image_data[band], band + 1)

    time.sleep(5)